In [1]:
from platform import python_version
print(python_version())

3.11.14


### BayesPrism

#### **Bayesian cell Proportion Reconstruction** Inferred using Statistical Marginalization (BayesPrism):

A Fully Bayesian Inference of Tumor Microenvironment composition and gene expression

BayesPrism consists of 
- the deconvolution modules and 
- the embedding learning module. 

The **deconvolution module** models a prior from cell type-specific expression profiles from scRNA-seq to jointly estimate the posterior distribution of cell type composition and cell type-specific gene expression from bulk RNA-seq expression of tumor (or non-tumor) samples. 

The **embedding learning** module uses Expectation-maximization (EM) to approximate the tumor expression using a linear combination of malignant gene programs while conditional on the inferred expression and fraction of non-malignant cells estimated by the deconvolution module.


#### Ref

Cell type and gene expression deconvolution with BayesPrism enables Bayesian integrative analysis across bulk and single-cell RNA sequencing in oncology

Chu, T. et al. & Danko, C.G.

https://www.nature.com/articles/s43018-022-00356-3


#### Concepts (paper)

Two layers of information are critical for understanding tumor composition: (1) the proportion of each cell type and (2) the levels of gene expression in each cell type. The rise of single-cell RNA sequencing (scRNA-seq) technologies has recently enabled direct, genome-wide measurement of the transcriptome in individual
cells within the TME and characterization of their heterogeneity. However, the cost of scRNA-seq and requirements for high-quality tissue limit the number of patient samples that can be assayed9. Moreover, **scRNA-seq is susceptible to technical biases in cell capture** 9 , which confound the recovery of cell type composition.

### Github

https://github.com/Danko-Lab/BayesPrism

- tutorial_deconvolution.html
- tutorial_embedding_learning.html


### Z (Posteior) and Gibbs distribution


Two different estimators, and the difference is mechanical.

#### **Fitted genes** (# 1918).

Gibbs sampling draws `Z[s,g,c]` from a posterior informed by *that sample's own bulk count for that gene*. If sample A's tumour genuinely expresses gene g more than sample B's, the data can say so. Where evidence is weak, the posterior shrinks toward the reference — but the shrinkage is evidence-dependent, and the estimate is compartment-specific.

#### **Projected genes** (the other #15432 = 17540-1918)
`full_Z` allocates the observed bulk count across compartments using weights built from `theta[s]` and the *fixed* reference `phi[g]`:

```
Z[s,g,c] ≈ X_bulk[s,g] · (theta[s,c] · phi[g,c]) / Σ_c' (theta[s,c'] · phi[g,c'])
```

Nothing in that expression carries sample-specific information about *which compartment* expresses gene g in sample s. Sample-to-sample variation has exactly two sources: variation in bulk expression, and variation in composition. The compartment assignment itself is the reference, applied identically to every sample.

**So yes, this can cause errors**, in a specific way. Take logs:

```
log Z[s,g,mal] − log Z[s,g,fib] = log(theta_mal[s]/theta_fib[s]) + log(phi[g,mal]/phi[g,fib])
```

A sample term plus a gene term, additive and separable. After compartment-CPM and per-gene centring, both cancel — meaning the malignant and fibroblast matrices should be **nearly identical for projected genes**. Any DE contrast run on them would return the same answer regardless of which compartment you think you're analysing.

Your `X_fib2` is 9959 genes, so ~85% are projected. Test it directly:

```python
g = X_mal2.columns.intersection(X_fib2.columns)
zm = ((X_mal2[g] - X_mal2[g].mean()) / X_mal2[g].std())
zf = ((X_fib2[g] - X_fib2[g].mean()) / X_fib2[g].std())
r = pd.Series({c: zm[c].corr(zf[c]) for c in g})

fitted = set(res.cell_type_expression(cmap["fibroblast"]).index)
print("fitted   genes, median r:", r[r.index.isin(fitted)].median().round(3))
print("projected genes, median r:", r[~r.index.isin(fitted)].median().round(3))
```

If projected genes come back near r=1 and fitted genes clearly below, then the 367 stroma hits and the 7 tumour hits are largely the same bulk contrast wearing two labels, and cross-compartment analyses should be restricted to the 1604 fitted genes.

I should flag that this is my reading of what `full_Z` must be doing — you wrote it, so check the actual allocation. If it does something more elaborate than reference-weighted splitting, the algebra changes.

### Single-cell quality: MAESTRO

MAESTRO (Model-based AnalysEs of Single-cell Transcriptome and RegulOme) is a Snakemake-based pipeline that processes single-cell RNA-seq and ATAC-seq data from raw FASTQ files through alignment, quality control, cell filtering, clustering, and cell-type annotation.

https://liulab-dfci.github.io/MAESTRO/

### TISCH2

https://tisch.compbio.cn/gallery/?cancer=PAAD&celltype=Acinar&celltype=Ductal&species=Human&treatment=None&primary=Primary


Peng J, Sun BF, Chen CY, Zhou JY, Chen YS, Chen H, Liu L, Huang D, Jiang J, Cui GS, Yang Y, Wang W, Guo D, Dai M, Guo J, Zhang T, Liao Q, Liu Y, Zhao YL, Han DL, Zhao Y, Yang YG, Wu W. Single-cell RNA-seq highlights intra-tumoral heterogeneity and malignant progression in pancreatic ductal adenocarcinoma. Cell Res. 2019 Sep;29(9):725-738. doi: 10.1038/s41422-019-0195-y. Epub 2019 Jul 4. Erratum in: Cell Res. 2019 Sep;29(9):777. doi: 10.1038/s41422-019-0212-1. PMID: 31273297; PMCID: PMC6796938.

### We have a problem !

> The reference is the hard half. Peng CRA001160 is symbol-indexed, and it's the pre-2018 symbols we just diagnosed.  
> Moving the df_bulk to ENSG doesn't help unless the reference moves too — otherwise the intersection goes to zero.  
> So you still need one symbol→ENSG mapping for the reference,  
> and it should use an archived Ensembl release (~92/93, contemporary with Peng),  
> not a current service, for the same reason the two-step round-trip was risky. 

#### ENSG IDs need normalising.

Peng J, Sun BF, Chen CY, Zhou JY, Chen YS, Chen H, Liu L, Huang D, Jiang J, Cui GS, Yang Y, Wang W, Guo D, Dai M, Guo J, Zhang T, Liao Q, Liu Y, Zhao YL, Han DL, Zhao Y, Yang YG, Wu W. Single-cell RNA-seq highlights intra-tumoral heterogeneity and malignant progression in pancreatic ductal adenocarcinoma. Cell Res. 2019 Sep;29(9):725-738. doi: 10.1038/s41422-019-0195-y. Epub 2019 Jul 4. Erratum in: Cell Res. 2019 Sep;29(9):777. doi: 10.1038/s41422-019-0212-1. PMID: 31273297; PMCID: PMC6796938.


In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

sys.path.insert(0, ROOT_SRC)


if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import create_dir
from libs.MTD_lib import MTD
from libs.cBioPortal_lib import cBioPortal
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config
from libs.prism_lib import PRISM
from libs.prism_program_lib import *
from libs.prism_diagnostics_helpers import *

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages
  warnings.warn(
/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'BRCA'
PSI_ID = 'ACC'
PSI_ID = 'CESC'
PSI_ID = 'PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/PAAD/config/all_lfc_cutoffs_PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [22]:
dstudy = 'scAtlas2025' # https://pubmed.ncbi.nlm.nih.gov/39636224/ - remove Peng and Metastasis
dstudy = 'Oh2023' # https://www.nature.com/articles/s41467-023-40895-6#Sec26
dstudy = 'Peng2018' # first study proposed by Claude

mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID, dstudy=dstudy,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", mtd.disease, case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD
>>> PAAD Tumor
>>> case Tumor
	DEGs 21270
		Up (#15231)
		Dw (#6039)

Up-regulated per biotype
                               biotype     n
0                            IG_C_gene     6
1                      IG_C_pseudogene     2
2                            IG_D_gene     7
3                            IG_V_gene     7
4                      IG_V_pseudogene    13
5                              Mt_rRNA     1
6                              Mt_tRNA     6
7                                  TEC   282
8                            TR_J_gene     1
9                            TR_V_gene    40
10                     TR_V_pseudogene    21
11                              lncRNA  6015
12                               miRNA   184
13                            misc_RNA   234
14              polymorphic_pseudogene    12
15                processed_pseudogene  3924
16                      protein_coding  2706
17      

In [23]:
cbio = cBioPortal(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

### Get all programs

In [24]:
verbose = False

df_psi = cbio.open_primary_site(verbose=verbose)
df_psi

,prog_id,cbioportal_study_id,active,gdc_project_id,psi_id,disease_id,disease_cd,primary_site,disease_context
0,TCGA,paad_tcga_pan_can_atlas_2018,True,TCGA-PAAD,PAAD,pancreatic_adenocarcinoma,PAAD,Pancreas,"TCGA pancreatic adenocarcinoma, PanCancer Atlas"
1,CPTAC3,paad_cptac_2021,True,CPTAC-3,PAAD,pancreatic_ductal_adenocarcinoma,PAAD,Pancreas,"CPTAC publication cohort, Cell 2021; 140 pancreatic cancers"
2,TCGA,skcm_tcga_pan_can_atlas_2018,True,TCGA-SKCM,SKCM,cutaneous_melanoma,SKCM,Skin,"TCGA skin cutaneous melanoma, PanCancer Atlas"
3,TCGA,brca_tcga_pan_can_atlas_2018,True,TCGA-BRCA,BRCA,breast_invasive_carcinoma,BRCA,Breast,"TCGA breast invasive carcinoma, PanCancer Atlas"
4,CPTAC2,brca_cptac_2020,True,CPTAC-2,BRCA,breast_cancer,BRCA,Breast,"CPTAC breast cancer publication cohort, Cell 2020"


### Open primary cites from cbio

In [25]:
PROG_ID = 'TCGA'
psi_id = 'PAAD'
psi_id = 'SKCM'
psi_id = 'BRCA'

PROG_ID = 'CPTAC2'
psi_id = 'BRCA'

PROG_ID = 'CPTAC3'
psi_id = 'PAAD'

_ = cbio.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, dstudy=dstudy, verbose=verbose)


### Prism Class instantitaion

https://github.com/Danko-Lab/BayesPrism

In [26]:
prism = PRISM(root0=ROOT0, root0_data=ROOT0_DATA)

verbose=True

prism.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, dstudy=dstudy, verbose=verbose)

prism.root_prism, prism.root_prism.exists()

Table opened ((7, 9)) at '/home/flavio/uv/perturb_agent/data/cbioportal_study_mapping.tsv'

-----------------------------
>> prog_id: CPTAC3
>> psi_id: PAAD
>> primary_site: Pancreas
>> disease_id: pancreatic_ductal_adenocarcinoma
>> disease_cd: PAAD

-----------------------------
>> cbioportal_study_id: paad_cptac_2021
>> gdc_project_id: CPTAC-3

---------- Bayes Prism -------------
>> root m.project: /home/flavio/uv/perturb_agent/data/multi_progs/PAAD
>> root d.study: /home/flavio/uv/perturb_agent/data/multi_progs/PAAD/Peng2018
>> root m.p.prism: /home/flavio/uv/perturb_agent/data/multi_progs/PAAD/Peng2018/prism
>> root m.p.lfc: /home/flavio/uv/perturb_agent/data/multi_progs/PAAD/Peng2018/lfc
>> root m.p.tahoe: /home/flavio/uv/perturb_agent/data/multi_progs/PAAD/Peng2018/tahoe

------- TCGA, CPTAC3, ... ------------
>> root disease: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD
>> root samples: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/samples
>> root lfc: /home/flavio/uv/pertu

(PosixPath('/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/Peng2018/prism'),
 True)

### 2. theta is now fixed -> expand Z to every gene

In [27]:
verbose=False
force=False

imax_tumor=250
imax_normal=50

exclude_prog_list=['CCLE']
disease_cd = 'PAAD'

_ = cbio.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, dstudy=dstudy, verbose=verbose)

dfn_tumor, dfn_normal, df_gtex, df_summ = cbio.get_all_data_from_disease(disease_cd=disease_cd, 
                                                           imax_tumor=imax_tumor, imax_normal=imax_normal,
                                                           exclude_prog_list=exclude_prog_list,
                                                           force=force, verbose=verbose)

Error reading csv/tsv '/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/Peng2018/lfc/expression_gtex_controls_counts.tsv': No columns to parse from file


In [28]:
print(dfn_tumor.shape)
dfn_tumor.head(3)

(60616, 134)


,geneid,symbol,biotype,T-C3L-02890,T-C3L-03635,T-C3L-02701,T-C3L-04072,T-C3L-00589,T-C3L-03123,T-C3N-01383,...,T-TCGA-HZ-7289,T-TCGA-3A-A9IN,T-TCGA-3A-A9IS,T-TCGA-2L-AAQM,T-TCGA-3A-A9IR,T-TCGA-3A-A9IV,T-TCGA-3A-A9IO,T-TCGA-2J-AABT,T-TCGA-H6-A45N,T-TCGA-3A-A9IJ
0,ENSG00000000003,TSPAN6,protein_coding,1486,2083,1558,546,1208,648,896,...,2506,442,38,299,77,158,394,659,1294,395
1,ENSG00000000005,TNMD,protein_coding,12,97,15,1,14,2,5,...,2,172,4,2,1,2,14,3,3,0
2,ENSG00000000419,DPM1,protein_coding,1330,1521,1499,986,1388,974,649,...,1638,940,1372,1008,1493,1059,922,839,717,1034


In [29]:
print(dfn_normal.shape)
dfn_normal.head(3)

(60616, 25)


,geneid,symbol,biotype,N-C3L-04072,N-C3L-00589,N-C3L-03123,N-C3L-04080,N-C3L-00640,N-C3N-01719,N-C3L-07033,...,N-C3N-03069,N-C3N-02765,N-C3L-07037,N-C3N-02589,N-C3N-02996,N-C3L-02606,N-C3N-03173,N-C3N-02696,N-TCGA-H6-8124,N-TCGA-H6-A45N
0,ENSG00000000003,TSPAN6,protein_coding,1633,1302,1079,1367,896,1188,1275,...,891,1063,1261,1821,554,1244,977,1576,3738,369
1,ENSG00000000005,TNMD,protein_coding,0,8,9,3,4,7,0,...,0,7,2,6,1,1,3,39,4,5
2,ENSG00000000419,DPM1,protein_coding,1281,937,655,956,1000,1504,1087,...,703,601,1059,1289,342,773,676,783,1532,1023


In [30]:
print(df_gtex.shape)
df_gtex.head(3)

(0, 0)


""


In [31]:
cbio.plot_boxplot_expression(dfn_tumor, do_log10=True, title = "Expression across tumor samples")

In [32]:
cbio.plot_boxplot_expression(dfn_normal, do_log10=True, title = "Expression across normal samples")

### Bulk - by geneid

In [33]:
force=False
verbose=True

df_bulk, df_meta = prism.build_bulk_matrix(dfn_tumor, dfn_normal, cbio.df_metadata, 
                                        keep_biotypes=("protein_coding", "lncRNA", "miRNA"),
                                        gene_key="geneid", force=force, verbose=verbose)

#--- reference geneid --> 
gene_map = prism.load_gene_map("geneid")
print(df_bulk.shape)
df_bulk.head(2)

Table opened ((27177, 153)) at '/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/Peng2018/prism/bulk_matrix_geneid.tsv'
Table opened ((153, 4)) at '/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/Peng2018/prism/bulk_metadata_geneid.tsv'
(27177, 153)


,T-C3L-02890,T-C3L-03635,T-C3L-02701,T-C3L-04072,T-C3L-00589,T-C3L-03123,T-C3N-01383,T-C3L-01124,T-C3L-00625,T-C3N-03439,...,N-C3N-03069,N-C3N-02765,N-C3L-07037,N-C3N-02589,N-C3N-02996,N-C3L-02606,N-C3N-03173,N-C3N-02696,N-TCGA-H6-8124,N-TCGA-H6-A45N
geneid,,,,,,,,,,,,,,,,,,,,,
ENSG00000000003,1486,2083,1558,546,1208,648,896,1532,821,1217,...,891,1063,1261,1821,554,1244,977,1576,3738,369
ENSG00000000005,12,97,15,1,14,2,5,6,3,11,...,0,7,2,6,1,1,3,39,4,5


In [34]:
cbio.plot_boxplot_expression(df_bulk, do_log10=True, title = "Expression across normal samples")

### Prism single-cell data reference - Peng 2019

load_cra001160.py  

Convert the Peng 2019 GSA deposit into an AnnData ready for
`paad_deconv.pseudobulk_reference()`.

Input (from ftp://download.big.ac.cn/gsa/CRA001160/):
- count-matrix.txt   2.77 GB dense TSV, genes x cells
- all_celltype.txt   2.1 MB, per-cell annotation

The matrix is dense text: ~20k genes x ~57k cells is ~1.1e9 values, which is 10-13 GB as a dense float array but well under 1 GB as CSR, since scRNA counts are >90% zeros. So it is parsed in row chunks and sparsified incrementally -- never materialised dense.

```Bash
lftp -e "cls -l; quit" ftp://download.big.ac.cn/gsa/CRA001160/

lftp -e "pget -n 8 -c count-matrix.txt; \
         get all_celltype.txt; get md5sum.txt; quit"      ftp://download.big.ac.cn/gsa/CRA001160/

In [35]:
prism.root_prism

PosixPath('/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/Peng2018/prism')

In [36]:
force=False
verbose=True

fname = "count-matrix.txt"
adata = prism.load_matrix(fname=fname, sep=' ', force=force, verbose=verbose)

fname_ad = fname.replace('.txt', '.h5ad')
filename_ad = prism.root_prism / fname_ad
compression = "gzip"


verbose=True
fname_celltype = "all_celltype.txt"
adata_ct = prism.attach_celltypes(adata=adata, fname_celltype=fname_celltype, verbose=verbose)

adata_ct

57,530 cells x 24,005 genes | obs: []
all_celltype.txt columns: ['cluster']
                             cluster
cell.name                           
T1_AAACCTGAGATGTCGG  Fibroblast cell
T1_AAACGGGGTCATGCAT    Stellate cell
T1_AAAGATGCATGTTGAC  Macrophage cell
using type_col='cluster'
barcode overlap: 57,530 / 57,530
cell_type
malignant             11315
Ductal cell type 1    10317
Endothelial cell       9117
Fibroblast cell        6742
Stellate cell          5907
Macrophage cell        5361
T cell                 3660
B cell                 2447
Acinar cell            1935
Endocrine cell          729
Name: count, dtype: int64


AnnData object with n_obs × n_vars = 57530 × 24005
    obs: 'cluster', 'cell_type', 'cell_state'

### Reference

In [ ]:
ref, s2t = prism.pseudobulk_reference(adata_ct)

print(ref.shape)
ref.head(3)

###  nnls_deconvolve()

It's the baseline cross-check — deliberately not part of the main path. It's there so you can ask "is my reference sane?" without trusting the engine you're validating.

What it computes. For each sample independently, it solves

min_w  ||Φᵀw − b_s||²    subject to  w ≥ 0

where b_s is the sample's CPM vector and Φᵀ is the genes × states signature matrix (each state row CPM-normalized). Then it rescales w to sum to 1. That's the dtangle / CIBERSORT family: linear unmixing under a Gaussian loss.

How it differs from prism_em, which matters more than it looks:

|	      | nnls_deconvolve	 |prism_em |
|---------|------------------|---------|
| loss	  | L2 on CPM	| multinomial on counts |
| gene weighting | high-expression genes dominate | Poisson variance weights each gene naturally |
| simplex	|  imposed post hoc by rescaling | enforced every iteration |
| malignant reference | fixed | sample-specific (stage 2) |

The L2-on-CPM part is the substantive difference. A gene at 5,000 CPM contributes ~10⁶× more residual than one at 5 CPM, so NNLS is effectively fit on a few dozen highly-expressed genes regardless of how informative they are. The multinomial likelihood weights each gene by its own expected count, which is the correct variance model for counts.

Why you'd actually run it. Concordance is a reference-quality diagnostic, and the pattern of disagreement is informative:

Stromal/immune states agree closely (Spearman > 0.8) → reference is fine
Stromal/immune states disagree → your phi is broken, or select_genes picked protocol-driven genes; fix that before interpreting anything
Malignant compartment disagrees while the rest agrees → expected and good. That's stage 2 doing its job. If NNLS and EM agree on purity, update_malignant_reference isn't contributing and PDAC classical/basal heterogeneity is still leaking into the stromal fractions

To wire it in:

In [ ]:
ref.head(2)

In [ ]:
len(res.genes), res.genes[:5]

In [ ]:
theta_res = res.theta
theta_res.head(2)

In [ ]:
res.genes[:5]

In [ ]:
df_bulk.head(2)

In [ ]:
g = df_bulk.index.intersection(pd.Index(res.genes))
len(df_bulk), len(res.genes), len(g)

In [ ]:
bulk_symbs = pd.read_csv(prism.root_prism / "bulk_matrix.tsv", sep="\t", index_col=0, usecols=[0])
print("CTGF in bulk_symbs:", "CTGF" in bulk_symbs.index, "| CCN2 in bulk_symbs:", "CCN2" in bulk_symbs.index)

In [ ]:
len(bulk_symbs), bulk_symbs[:5]

In [ ]:
df_bulk.head(3)

In [ ]:
ref.head(3)

### ref_new --> new reference, by geneid (ensbeml)

In [ ]:
force=False
verbose=True

# ref, s2t = prism.pseudobulk_reference(adata_ct)

ref_new, df_to_from = prism.harmonize_reference_to_ensembl(bulk_symbs=bulk_symbs, ref=ref, gene_map=gene_map, force=force, verbose=verbose)

print(ref_new.shape)
print(df_to_from.status.value_counts())
ref_new.head(2)

In [ ]:
df_to_from[df_to_from.renamed].head(20)

In [ ]:
df_to_from2 = df_to_from[~pd.isnull(df_to_from.geneid)]
df_to_from2

In [ ]:
gene_subset = prism.select_genes(ref_new)
len(gene_subset), gene_subset[:5]

In [ ]:
print(type(s2t), len(s2t))
s2t

### Removing bad columns - see mtp_cBIO_66_PAAD_batch_and_samples_to_remove

In [ ]:
cols = df_bulk.columns
cols

### Maligant Cluster

In [ ]:
import importlib, libs.prism_malig_lib as pml
importlib.reload(pml)
print(pml.__version__)

In [ ]:
df_bulk.shape

In [ ]:
# keep = res.theta.index.to_list()
# df_bulk = df_bulk[keep]
# df_bulk.shape

In [ ]:
root_mprog_disease = cbio.root_mprog_disease
mal_cell_name = "Ductal cell type 2"

mc = pml.MalignantCluster(prism=prism, res=res, 
                          df_bulk=df_bulk, ref=ref_new, 
                          root_mprog_disease = root_mprog_disease,
                          mal_cell_name = mal_cell_name,
                          organ="Pancreas")

mc

In [ ]:
tum_cols = [col for col in cols if col.startswith("T-")]
norm_cols = [col for col in cols if col.startswith("N-")]

nd = mc.df_theta.loc[norm_cols, ["Ductal cell type 2","Acinar cell","Ductal cell type 1"]].round(3)
nd.sort_values("Ductal cell type 2")

In [ ]:
MIN_SHARE, MIN_COUNTS = 0.3, 10

cmap = {
    "malignant":  "Ductal cell type 2",
    "fibroblast": "Fibroblast cell",     # whatever Peng calls stellate/CAF
    "macrophage": "Macrophage cell",
    "endothelial": "Endothelial cell",
    "acinar": "Acinar cell",
}

batch = pd.Series(np.where(mc.df_theta.index.str.contains("TCGA"), "TCGA", "CPTAC"),
                  index=mc.df_theta.index, name="cohort")

X_fib2 = mc.compartment_matrix(cmap["fibroblast"], min_share=MIN_SHARE, min_counts=MIN_COUNTS,
                               batch=batch, samples=tum_cols)

zeros = (mc.df_theta.loc[tum_cols] == 0).sum(axis=1)
print(zeros.value_counts().sort_index().to_dict())

bad_tumors = mc.df_theta.loc[tum_cols][zeros >= 3].T
bad_tumors

### Bad Normal samples

In [ ]:
bad_cols1 = nd[(nd['Acinar cell'] == 0) | (nd['Acinar cell'].isnull()) |
               (nd['Ductal cell type 1'] == 0) | (nd['Ductal cell type 1'].isnull()) ].index.to_list()

### Bad Tumor samples

In [ ]:
tumor_samples = [s for s in mc.df_theta.index if s.startswith("T-")]
failed = mc.df_theta.index[mc.df_theta.isna().all(axis=1)
                           | (mc.df_theta.max(axis=1) > 0.98)].tolist()
bad_cols2 = [s for s in tumor_samples if s in failed]

np.array(bad_cols2)

In [ ]:
bad_cols = bad_cols1 + bad_cols2
print(f"There are {len(bad_cols)} samples")
np.array(bad_cols)

### calc bayesprism - again, using ref_new

In [ ]:
print(df_bulk.shape)
df_bulk.head(2)

In [ ]:
good_cols = [x for x in df_bulk.columns if x not in bad_cols]

df_bulk_good = df_bulk[good_cols]
print(df_bulk.shape, df_bulk_good.shape)
df_bulk_good.head(2)

In [ ]:
verbose=True
force=False

meta_desc = dict(reference="Peng2019_CRA001160",
              cohorts=["TCGA-PAAD", "CPTAC3"],
              strand="unstranded",
              method="InstaPrism")

gene_subset = prism.select_genes(ref_new)

good_cols = [x for x in df_bulk.columns if x not in bad_cols]
df_bulk_good = df_bulk[good_cols]

res = prism.run_bayesprism(df_bulk=df_bulk_good, ref=ref_new, state_to_type=s2t,
                           gene_subset=gene_subset, meta_desc=meta_desc, force=force, verbose=verbose)

type(res)

In [ ]:
# force to avoid mistakes
df_bulk = df_bulk_good

th_nnls = prism.nnls_deconvolve(df_bulk, ref_new, genes=res.genes)
print(th_nnls.shape)
th_nnls

In [ ]:
cbio.plot_boxplot_expression(df_bulk, do_log10=True, title = "Expression across normal samples")

### Massive improvement 

the normals are gone and NNLS now returns a real composition. So `ref_new` fixed a genuine bug, not just a naming issue. Worth noting *why*: with the old `ref`, the symbol mismatch meant a large fraction of the design matrix was misaligned, and NNLS collapsed to the boundary. That's a stronger validation of the harmonisation than the gene counts were.

But the agreement is stratified in a way that should worry you.

**High-abundance compartments agree** (Acinar 0.96, Fibroblast 0.93, Ductal-2 0.86). **Low-abundance ones don't** (T cell 0.13, Macrophage 0.14, B cell 0.16, Endothelial 0.19). Spearman ~0.15 is essentially no agreement — two methods on the same data producing unrelated rankings.

That's not a tie-breaking situation; it means **the immune and endothelial fractions are not identified by this data**. Which retroactively explains a lot: those were exactly the compartments whose couplings didn't replicate, that ran at n=27–39 after `min_theta`, and that I flagged as "clean" on the Levene test in a way I suspected was uninformative. It was.

The two large disagreements are also worth flagging:

- **Ductal-2: EM 0.34 vs NNLS 0.087** — a 4× difference in *tumour purity*. That's the quantity everything downstream conditions on.
- **Fibroblast: EM 0.37 vs NNLS 0.57**, in the opposite direction. Ductal-2 and Fibroblast trading mass is the classic signature of two compartments the model can't separate — consistent with your `zm`/`zf` correlation of 0.927.


In [ ]:
theta_res = res.theta

conc = pd.DataFrame({
    "spearman": {k: theta_res[k].corr(th_nnls[k], method="spearman") for k in res.states},
    "mean_em":   theta_res.mean(),
    "mean_nnls": th_nnls.mean(),
})
conc["bias"] = conc.mean_em - conc.mean_nnls
print(conc.sort_values("spearman"))

### Old review by Claude/Opus 5

NNLS isn't a gold standard (no priors, no shrinkage, sensitive to collinearity), so I'd trust the EM estimate. But the disagreement bounds what you can claim.

That last one is the diagnostic: if EM's malignant fraction correlates with NNLS's fibroblast fraction, the two methods are splitting the same mass differently, and purity is method-dependent rather than measured.

Practical upshot for the writeup: report θ agreement per compartment, restrict compartment-level claims to Ductal-2, Fibroblast and Acinar, and treat immune/endothelial results as exploratory at best.

### New review by Claude/Opus 5 - 2026-08-26

Removing the 19 samples barely moved the concordances — fibroblast 0.932 → 0.921, acinar 0.960 → 0.961, Ductal-2 0.864 → 0.850. So the collapsed samples weren't driving the rank agreement; the estimator disagreement is a property of the whole cohort. The immune compartments stayed poor (T cell 0.14, endothelial 0.15, macrophage 0.19, B cell 0.22).

**The bias column is the part that should concern you.** Ductal-2: EM 0.340 vs NNLS 0.077 — a 4.4× discrepancy in mean malignant fraction. Fibroblast: EM 0.370 vs NNLS 0.624, a 1.7× discrepancy the other way. Those are the two compartments your entire analysis rests on, and the two estimators disagree by more than the biological range you're trying to resolve. Rank order agrees well (0.92, 0.85); absolute level does not, and they trade mass between exactly the two compartments.

That has a direct consequence for the `min_share=0.3` filter, which is a threshold on *level*, not rank. Under NNLS's fibroblast mass, many more genes would clear it; under EM's, fewer do. Your gene selection is estimator-dependent in a way the concordance table's high Spearman conceals.

**Worth knowing which is closer to truth.** NNLS says PDAC is 62% fibroblast and 8% malignant; EM says 37% and 34%. Published pathologist estimates for PDAC purity typically fall between those, though closer to the low end. Neither is obviously right, but the gap means "malignant fraction" in your analysis is not a well-identified quantity.

For the two things that actually matter downstream:

- **Program scores** are computed within-compartment and are θ-orthogonal after residualization, so a level bias mostly doesn't propagate.
- **`min_share`** is directly exposed to it. Worth checking sensitivity — rerun the fibroblast matrix at `min_share` ∈ {0.1, 0.2, 0.3} and see how much the gene set and the fib×mal correlation move. If the answer is stable, the level ambiguity doesn't matter for your conclusions. If it moves a lot, `min_share` needs to be justified against something other than a number that two estimators disagree about by 1.7×.

### res.theta was the old one

Nearly identical to the stale version — Ductal-2 0.850 both times, fibroblast 0.921 → 0.929, acinar 0.961 → 0.962. Means shifted a little (Ductal-2 0.340 → 0.301, fibroblast 0.370 → 0.399) which is just the 19 dropped samples, mostly normals with extreme values.

So nothing in my read changes. The structural finding stands:

**Two groups.** Fibroblast, acinar, Ductal-2 at ρ = 0.85–0.96 — rank order reproducible under a completely different estimator. T cell, endothelial, macrophage, B cell at 0.14–0.22 — not measuring the same quantity. Endocrine 0.30, stellate 0.40 in between.

**The level disagreement is the unresolved part.** Ductal-2: 0.30 vs 0.077 (3.9×). Fibroblast: 0.399 vs 0.624. And they trade in opposite directions with nearly equal magnitude (+0.225 / −0.225), which is the two compartments competing for the same reads. Note stellate also disagrees in the fibroblast direction (−0.043), so it's plausibly a three-way ambiguity among stroma-like references.

For your downstream work the practical consequence remains `min_share`, which thresholds on level. Under NNLS's fibroblast mass the filter would admit far more genes than under EM's. Worth testing directly:

If the correlation is stable across that range, the level ambiguity doesn't reach your conclusions and you can stop worrying about it. If it moves substantially, `min_share=0.3` is a choice resting on a quantity two estimators disagree about by a factor of 1.6.

In [ ]:
print(df_bulk.shape)
df_bulk.head(2)

In [ ]:
keep = df_bulk.columns.to_list()

for ms in [0.1, 0.2, 0.3]:
    Xf = mc.compartment_matrix(cmap["fibroblast"], min_share=ms, min_counts=MIN_COUNTS, batch=batch, samples=keep)
    Xm = mc.compartment_matrix(cmap["malignant"], min_share=ms, min_counts=MIN_COUNTS, batch=batch, samples=keep)
    r, ns, ng = gene_corr(Xf, Xm)
    print(f"min_share={ms}: {Xf.shape[1]:>6} fib genes, fib x mal r {r.median():.3f} ({ng} shared)")

Stable. 0.912 → 0.904 → 0.879 across a 3× range of `min_share`, while the shared gene count halves (12,030 → 6,101). So the coupling isn't an artifact of the filter, and the level ambiguity between EM and NNLS doesn't reach it. That question is closed.

But note the numbers went **up**, not down. You removed 19 samples including 4 collapsed tumors and all normals, and fib×mal at `min_share=0.3` went 0.869 → 0.879. The gene counts also jumped — 11,156 fibroblast genes now versus 10,853 before, on fewer samples. Both are the filters behaving differently on a cleaner sample set (`min_share` requires clearing the threshold in half the samples; drop the low-θ normals and more genes qualify).

So: after batch correction, normals removed, collapsed tumors removed, and across three filter settings, two compartments correlate per-gene at ~0.90 while each correlates with bulk at ~0.75. The coupling has survived every attempt to explain it away.

The one structural explanation left is the one I named earlier and never tested: `Z_full` partitions a single `X_bulk[s,g]` across compartments, so `Z[s,g,fib]` and `Z[s,g,mal]` are two shares of the same number. Within-compartment CPM removes the scale but not the common origin. That would be inherent to BayesPrism rather than a bug, and it predicts the coupling should be weaker for genes where the compartments' shares vary independently across samples.

Testable with what you already have:

```python
sh = cns.share  # fibroblast share per sample x gene
r_fm, _, _ = gene_corr(Xf, Xm)
share_var = sh.reindex(columns=r_fm.index).std()
print("gene-level r vs SD of fibroblast share:",
      round(r_fm.corr(share_var, method="spearman"), 3))
```

If genes whose share varies a lot across samples show lower cross-compartment r, the partition explanation holds. If there's no relationship, it doesn't, and I'd stop proposing mechanisms — the honest statement is that two compartments co-vary at 0.90 for reasons not identified, and any gene-level cross-compartment claim needs that caveat attached.

In [ ]:
from scipy.spatial.distance import jensenshannon
js = pd.Series({s: jensenshannon(theta_res.loc[s], th_nnls.loc[s]) for s in theta_res.index})
js.hist()


### jensenshannon distance between theta_res x th_nnls


Good diagnostic — it puts the level disagreement on a per-sample footing instead of a per-compartment mean.

**The mode sits at 0.30–0.35.** With `scipy`'s `jensenshannon` returning the square-root distance (bounded [0,1] for base-e... actually [0,1] only with base 2 — worth checking, since `scipy` defaults to base e and the bound is then √ln2 ≈ 0.83). Either way, the typical sample sits at roughly a third of the maximum possible divergence between two composition vectors. That's substantial disagreement for the same data under two estimators, and it's consistent with the Ductal-2/fibroblast mass trade you saw in the means.

**The distribution is unimodal with a right tail**, not bimodal. Bulk of samples 0.15–0.45, then a handful at 0.55–0.80. So there isn't a clean "good samples / failed samples" split — disagreement is pervasive, with a few extreme cases. The tail is worth naming:

```python
print(js.nlargest(8).round(3).to_dict())
print("mean JS by cohort:", js.groupby(batch.reindex(js.index)).mean().round(3).to_dict())
print("JS vs theta_mal:", round(js.corr(theta_res["Ductal cell type 2"]), 3))
```

Two things I'd want from that: whether the tail is the low-purity samples (where less information is available to identify composition), and whether CPTAC and TCGA disagree at different rates — which would connect this back to the cohort structure in θ.

**What it doesn't change.** JS is dominated by the large compartments, so this is mostly restating the Ductal-2/fibroblast trade. And your `min_share` sweep already showed the fib×mal coupling is insensitive to the level ambiguity. So this is a caveat about reporting absolute compositions, not a threat to the within-compartment analyses.

One methodological note: `js` is computed on `theta_res.index`, which after the refresh may still include the samples you dropped. If the 0.55–0.80 tail turns out to be normals or collapsed tumors, restrict to `keep` before interpreting.

In [ ]:
print(js.nlargest(8).round(3).to_dict())
print("mean JS by cohort:", js.groupby(batch.reindex(js.index)).mean().round(3).to_dict())
print("JS vs theta_mal:", round(js.corr(theta_res["Ductal cell type 2"]), 3))

Six of the top eight are `TCGA-3A-*`. That's a single submitting site — TCGA tissue source site 3A is one institution — and it accounts for the entire extreme tail. That's not a property of tumors; it's a property of where they came from.

So the JS tail is site-specific, which reframes the other two numbers:

**Cohort difference (0.295 vs 0.341)** is small and probably driven by these same 3A samples. Check by excluding them:

```python
is3a = js.index.str.contains("TCGA-3A-")
print("TCGA mean JS excluding 3A:", round(js[~is3a & (batch.reindex(js.index)=="TCGA")].mean(), 3))
print("n 3A samples:", is3a.sum())
```

If TCGA drops to ~0.30, the cohort effect on estimator disagreement disappears entirely and it's one site.

**JS vs θ_malignant = 0.455** is the more interesting relationship, but it's likely confounded by the same thing — the 3A samples appeared in your earlier `nlargest` on θ (3A-A9IS at 0.974, 3A-A9IR at 0.971). So high-purity samples disagree more, and the high-purity samples are disproportionately 3A. Partial it out:

```python
print("JS vs theta_mal, excluding 3A:",
      round(js[~is3a].corr(theta_res.loc[js.index[~is3a], "Ductal cell type 2"]), 3))
```

If the correlation survives at ~0.4, it's a real property of the estimators — as θ_malignant rises, EM and NNLS diverge, which fits the mass-trade pattern. If it collapses, it was 3A.

**What to do about 3A.** Worth looking at what's distinctive:

```python
th3 = theta_res.loc[is3a].mean().round(3)
thr = theta_res.loc[~is3a].mean().round(3)
print(pd.DataFrame({"3A": th3, "rest": thr, "diff": (th3-thr).round(3)}).to_string())
print(df_bulk.loc[js.index[is3a]].sum(axis=1).describe())
```

A site-level effect on composition estimates is a third batch layer under the TCGA/CPTAC split you already corrected — and your current `batch` Series treats all TCGA as one level. If 3A is systematically different, it deserves its own level, which is a one-line change to how you build `batch`.

### "Ductal cell type 2" x "Fibroblast cell"

That refutes my mass-swapping hypothesis. r = −0.0005 between EM's malignant and NNLS's fibroblast is as close to zero as a correlation gets — the two methods are **not** trading mass between those compartments.

So the picture is better than I suggested. Both compartments have high rank agreement (0.86, 0.93), and the disagreement is a **scale/offset difference, not a reordering**. NNLS systematically puts less mass in Ductal-2 (0.087 vs 0.34) and more in Fibroblast (0.57 vs 0.37), but the *sample ordering* is preserved in both.

That's a much less damaging failure mode, and it explains the JS distance: JS is sensitive to absolute proportions, so a consistent offset inflates it even when the ranking is intact.

Practically, this means:

- **Anything rank-based is safe** — Spearman couplings, `axis_modality`, cohort comparisons. Your `prolif`↔`iCAF` finding uses Spearman, so it's unaffected.
- **Anything using θ as an absolute number is not** — `min_theta=0.02` cutoffs, the purity decoupling regression, and any statement like "median tumour purity was 0.34."

The offset direction is worth understanding, though I'd note NNLS is the less trustworthy estimator here: no priors, no shrinkage, and highly sensitive to collinear compartments. BayesPrism's EM with the reference prior is the better estimate, and its Ductal-2 median of ~0.31 is consistent with published PDAC purity, whereas NNLS's 0.087 is implausibly low for a tumour cohort.

One thing worth checking, since it's cheap and would be reassuring:

```python
theta_res["Ductal cell type 2"].corr(th_nnls["Ductal cell type 2"], method="pearson")
np.polyfit(th_nnls["Ductal cell type 2"], theta_res["Ductal cell type 2"], 1)
```

If Pearson is also high and the slope is a clean constant, the relationship is affine and you can say the two agree up to a scaling — which is a clean, statable result rather than a caveat.

### Is normal like - use Pearson

In [ ]:
for k in ["Ductal cell type 2", "Fibroblast cell"]:
    print(f"{k:20}: {theta_res[k].corr(th_nnls[k], method='spearman'):.3f}")
print("\nMal x Fib NNLS corr")
print(th_nnls["Ductal cell type 2"].corr(th_nnls["Fibroblast cell"], method="spearman"))
print("\nMal x Fib Theta corr")
print(theta_res["Ductal cell type 2"].corr(theta_res["Fibroblast cell"], method="spearman"))

In [ ]:
print("\nMal x Fib NNLS corr")
print(th_nnls["Ductal cell type 2"].corr(th_nnls["Fibroblast cell"]).round(3))
print("\nMal x Fib Theta corr")
print(theta_res["Ductal cell type 2"].corr(theta_res["Fibroblast cell"]).round(3))

In [ ]:
print(theta_res["Ductal cell type 2"].isna().sum(), th_nnls["Ductal cell type 2"].isna().sum())
print(theta_res.isna().all(axis=1).sum(), "samples all-NaN in EM")
theta_res.index[theta_res["Ductal cell type 2"].isna()][:10]

In [ ]:
theta_res["Ductal cell type 2"].corr(th_nnls["Ductal cell type 2"], method="pearson")
np.polyfit(th_nnls["Ductal cell type 2"], theta_res["Ductal cell type 2"], 1)

In [ ]:
theta_res["Ductal cell type 2"]

In [ ]:
th_nnls["Ductal cell type 2"]

### why Zfull resulted in 17540 genes?

Because full_Z reconstructs the full gene set, not the subset BayesPrism fitted on.

The three numbers you've seen trace it:

- 17540 — genes in full_Z, the whole expression matrix
- 1918 — genes in cell_type_expression, the marker-based fit ~ res.Z.shape
- ~15000 — the complement

BayesPrism runs on gene_subset (marker/signature genes) for tractability and identifiability. That gave 1918. 

full_Z then projects the remaining ~15000 genes onto the fitted compartment basis  
which is precisely why you built it: to recover the lncRNA/antisense loci (FAM83A-AS1, HOXA10-AS, HOXB-AS3/4, MIR7-3HG) that the marker fit excluded.

Shape is (153 samples, 10 cell types, 17476 genes) — build_ms_from_full_Z resolves that axis order automatically.

The consequence you should hold onto: 
- those ~15000 recovered genes are not Gibbs posterior estimates. 
- they're projections onto a basis fitted from 1918 genes, 
- so their sampling variance is structurally different 
  - no posterior shrinkage in the same sense, 
  - and their between-sample variation partly reflects the projection rather than compartment-specific evidence.

In [ ]:
res.Z.shape, df_bulk.shape, ref_new.shape

In [ ]:
Zfull, gfull = prism.full_Z(res, df_bulk, ref_new)
print(Zfull.shape)

In [ ]:
ref_new.shape

In [ ]:
dic = {}

for cell_state in res.states:
    Z = prism.state_expression(Zfull, gfull, res, cell_state)
    dic[cell_state] = Z
    print(f"{cell_state:<20} dim {Z.shape}")


In [ ]:
i=0
key = list(dic.keys())[i]

print(key)
dic[key].head(3).T

### Ductal 2 - malignant

In [ ]:
# has normal samples
Zmal = prism.state_expression(Zfull, gfull, res, "Ductal cell type 2")
print(Zmal.shape)
Zmal.head(3).T

In [ ]:
gfull[:3]

In [ ]:
df_to_from2.head(3)

In [ ]:
print(df_bulk.shape)
df_bulk.head(3)

In [ ]:
prog1 = ["FAM83A-AS1", "HOXA10-AS", "HOXB-AS3", "HOXB-AS4", "MIR7-3HG"]
prog2 = ["GATA6", "KRT17", "NEAT1", "H19", "DLEU1", "DLEU2"]

for g in prog1:
    row = df_to_from2[df_to_from2['current_symbol'] == g]
    if row.empty:
        print(f"Gene {g} not found in df_to_from2")
    else:
        geneid = row.iloc[0].geneid
        print(g, geneid, geneid in df_bulk.index.to_list())

### survived build_bulk_matrix?

> Almost certainly df_bulk is the culprit: build_bulk_matrix defaults to keep_biotypes=("protein_coding",), which removes every lncRNA. Rebuild with them included:

In [ ]:
def find_geneid(symbol:str):
    row = df_to_from2[df_to_from2['current_symbol'] == symbol]
    if row.empty:
        return None, False
    else:
        geneid = row.iloc[0].geneid
        return row.iloc[0].geneid, geneid in df_bulk.index.to_list()
        

{g: find_geneid(g) for g in prog1}

In [ ]:
ref_new.head(3)

In [ ]:
gene = "HOXB-AS4"

print("in bulk_symbs gene_map:", (gene_map["symbol"] == gene).any())
print(f"{gene} in ref_new (raw):", gene in ref_new.columns)
print(f"{gene} in df_to_from:", (df_to_from2["ref_symbol"] == gene).any())

In [ ]:
{g: find_geneid(g) for g in prog2}

### Confirm

NEAT1, H19 and DLEU2 missing is a much stronger signal than HOXB-AS4 was. These aren't obscure — NEAT1 and H19 are among the most abundant lncRNAs in any tissue, H19 is a classic PDAC lncRNA, and NEAT1 is the paraspeckle scaffold. If a reference lacks those, the gap isn't about low abundance.

The likely cause is nuclear retention. NEAT1 and H19 are predominantly nuclear, and Peng used 10x 3' whole-cell scRNA-seq — cytoplasmic-biased, so nuclear-retained transcripts are systematically under-recovered. DLEU2 is the same class. Note DLEU1 survived while DLEU2 didn't, which is consistent with transcript-level capture differences at the same locus.

Confirm it's the reference and not the mapping:

In [ ]:
for g in ["NEAT1","H19","DLEU2","MALAT1","XIST","KCNQ1OT1","MEG3"]:
    print(f"{g:10s} bulk_symbs:{(gene_map['symbol']==g).any()!s:5s} ref_new:{g in ref_new.columns}")

### Ductal 2 - malignant

In [ ]:
# has normal samples
Zmal = prism.state_expression(Zfull, gfull, res, "Ductal cell type 2")
print(Zmal.shape)
Zmal.head(3)

In [ ]:
ok = np.sum([1 if geneid in ref_new.columns.to_list() else 0 for geneid in df_bulk.index])
ok, len(df_bulk), ref_new.shape[1]

In [ ]:
bt = gene_map.reindex(df_bulk.index)["biotype"]
pd.crosstab(bt, df_bulk.index.isin(ref_new.columns), normalize="index").round(3)

In [ ]:
ref_new.head(3)